# Pipeline 02: Style Profiling + Region Detection

**Goal:** Two capabilities that make the editor production-ready:

1. **Style Profiling** — Analyze paired edits to build scene-aware style profiles
2. **Region Detection (SAM)** — Segment faces, sky, subjects for regional editing

**Depends on:** Pipeline 01 (for pair analysis schema), `shared.py`

**Output:** `style_profiles.json`, `mask_labels_cache.json`

**Runtime:** Cells 1-5 (profiling) = CPU fine. Cells 6+ (SAM) = GPU required (T4 minimum).

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/photo-style-rl'

!pip install anthropic pillow numpy matplotlib opencv-python-headless -q

import shutil
shutil.copy(f'{PROJECT}/src/shared.py', '/content/shared.py')

import json
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import userdata
import anthropic

from shared import (
    PROJECT, RAW_DIR, EDITED_DIR, CHECKPOINTS_DIR, SCENE_TYPES,
    pair_files, image_to_base64, extract_json,
    get_mask_properties, classify_mask_heuristic, classify_masks_with_llm,
    segment_and_label, resize_for_sam, visualize_segments,
    DeterministicRenderer, feather_mask, apply_regional_edits,
    save_style_profile, show_before_after
)

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
pairs = pair_files()
print(f"Loaded {len(pairs)} matched pairs")

## 1. Comprehensive Style Profiling

Analyze a stratified sample of pairs. For each pair, Claude estimates Lightroom parameters. We aggregate into:
- **Global style defaults** — your average editing tendency
- **Scene-specific profiles** — how your style changes for night vs daylight vs portraits
- **Confidence intervals** — how consistent you are per parameter

This becomes the auto-edit engine: classify scene → look up profile → apply.

In [ ]:
# Batch style analysis (with incremental saves and rate limit handling)
import time

def analyze_edit_for_profile(raw_path, edited_path, client):
    """Claude analyzes a raw/edited pair and estimates scene type + parameters."""
    raw_img = Image.open(raw_path).convert('RGB')
    edited_img = Image.open(edited_path).convert('RGB')
    raw_b64 = image_to_base64(raw_img)
    edited_b64 = image_to_base64(edited_img)

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=2000,
        system=f"""You are an expert photo editor analyzing before/after pairs.
Estimate the Lightroom adjustments applied. Respond with ONLY valid JSON.

Required fields:
- "scene_type": one of {json.dumps(SCENE_TYPES)}
- "style_description": natural language summary
- "global": object with parameter names and numeric values (only non-zero)
- "tone_curve": object with {{blacks, shadows, midtones, highlights, whites}} if relevant
- "color_mixer": object with color names (reds, oranges, yellows, greens, aquas, blues, purples, magentas) and {{h, s, l}} shifts if relevant

Output ONLY JSON.""",
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": raw_b64}},
                {"type": "text", "text": "RAW (before)."},
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": edited_b64}},
                {"type": "text", "text": "EDITED (after). Output ONLY JSON."},
            ]
        }],
    )

    result = extract_json(response.content[0].text)
    if result is None:
        raise ValueError(f"Could not parse: {response.content[0].text[:200]}")
    return result


# Stratified sample of 30 pairs
sample_size = 30
sample_indices = np.linspace(0, len(pairs)-1, sample_size, dtype=int)
analyses_path = f'{CHECKPOINTS_DIR}/style_profile_analyses.json'

# Resume from existing analyses
if os.path.exists(analyses_path):
    with open(analyses_path, 'r') as f:
        analyses = json.load(f)
    analyzed_raws = {a['raw'] for a in analyses}
    print(f"Resuming: {len(analyses)} already analyzed")
else:
    analyses = []
    analyzed_raws = set()

print(f"Analyzing {sample_size} pairs for style profiling...\n")

for i, idx in enumerate(sample_indices):
    raw_f, edited_f = pairs[idx]

    if raw_f in analyzed_raws:
        print(f"  [{i+1}/{sample_size}] {raw_f} — cached, skipping")
        continue

    print(f"  [{i+1}/{sample_size}] {raw_f} → {edited_f}...", end=" ")

    try:
        analysis = analyze_edit_for_profile(
            os.path.join(RAW_DIR, raw_f),
            os.path.join(EDITED_DIR, edited_f),
            client,
        )
        analyses.append({"raw": raw_f, "edited": edited_f, "analysis": analysis})
        scene = analysis.get('scene_type', 'unknown')
        desc = analysis.get('style_description', '')[:50]
        print(f"[{scene}] {desc}")

        # Incremental save — never lose progress
        with open(analyses_path, 'w') as f:
            json.dump(analyses, f, indent=2)

    except Exception as e:
        print(f"Error: {e}")
        # Basic rate limit handling
        if "rate" in str(e).lower() or "429" in str(e):
            print("  Rate limited, waiting 30s...")
            time.sleep(30)

print(f"\nTotal analyses: {len(analyses)}")

In [ ]:
# Aggregate analyses into scene-specific and overall style profiles
scene_profiles = {}

for a in analyses:
    analysis = a['analysis']
    scene = analysis.get('scene_type', 'unknown')

    if scene not in scene_profiles:
        scene_profiles[scene] = {'params': {}, 'count': 0, 'descriptions': []}

    scene_profiles[scene]['count'] += 1
    scene_profiles[scene]['descriptions'].append(analysis.get('style_description', ''))

    # Collect all numeric params
    for group_key in ['global', 'tone_curve']:
        if group_key in analysis:
            for key, value in analysis[group_key].items():
                if isinstance(value, (int, float)):
                    full_key = f"tone_curve.{key}" if group_key == 'tone_curve' else key
                    if full_key not in scene_profiles[scene]['params']:
                        scene_profiles[scene]['params'][full_key] = []
                    scene_profiles[scene]['params'][full_key].append(value)

    # Color mixer
    if 'color_mixer' in analysis:
        for color, shifts in analysis['color_mixer'].items():
            if isinstance(shifts, dict):
                for axis, val in shifts.items():
                    if isinstance(val, (int, float)):
                        full_key = f"color_mixer.{color}.{axis}"
                        if full_key not in scene_profiles[scene]['params']:
                            scene_profiles[scene]['params'][full_key] = []
                        scene_profiles[scene]['params'][full_key].append(val)

# Print summary
print("=" * 70)
print("YOUR STYLE PROFILES BY SCENE TYPE")
print("=" * 70)

all_params = {}
for scene, profile in sorted(scene_profiles.items(), key=lambda x: -x[1]['count']):
    print(f"\n{'─' * 50}")
    print(f"Scene: {scene.upper()} ({profile['count']} photos)")
    print(f"{'─' * 50}")

    for key, values in sorted(profile['params'].items()):
        mean = np.mean(values)
        std = np.std(values)
        if abs(mean) > 1:
            print(f"  {key:<30} mean={mean:+6.1f}  std={std:5.1f}")

        if key not in all_params:
            all_params[key] = []
        all_params[key].extend(values)

# Overall
print(f"\n{'=' * 70}")
print(f"OVERALL STYLE (across {len(analyses)} photos)")
print(f"{'=' * 70}")

overall_profile = {}
for key, values in sorted(all_params.items()):
    mean = np.mean(values)
    if abs(mean) > 1:
        print(f"  {key:<30} mean={mean:+6.1f}")
        overall_profile[key] = {"mean": round(mean, 1), "std": round(float(np.std(values)), 1)}

# Save
style_profiles = {
    "overall": overall_profile,
    "by_scene": {
        scene: {
            "count": p['count'],
            "params": {
                k: {"mean": round(np.mean(v), 1), "std": round(float(np.std(v)), 1)}
                for k, v in p['params'].items() if abs(np.mean(v)) > 1
            }
        }
        for scene, p in scene_profiles.items()
    }
}

path = save_style_profile(style_profiles, 'style_profiles_llm.json')
print(f"\nSaved to {path}")
print(f"Scenes: {list(scene_profiles.keys())}")

## 2. Region Detection with SAM

SAM segments the image into pixel-perfect masks. Combined with Claude for semantic labeling, this enables: "brighten just the face," "cool the sky," "add clarity to the ground."

In [ ]:
# SAM setup
!pip install segment-anything -q

import torch
import cv2
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

SAM_CHECKPOINT = f'{CHECKPOINTS_DIR}/sam_vit_h_4b8939.pth'

if not os.path.exists(SAM_CHECKPOINT):
    print("Downloading SAM ViT-H (~2.5 GB)...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O {SAM_CHECKPOINT}
    print("Done.")
else:
    print(f"SAM checkpoint exists at {SAM_CHECKPOINT}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cpu':
    print("⚠️ SAM on CPU will be very slow. Switch to GPU runtime.")

sam = sam_model_registry['vit_h'](checkpoint=SAM_CHECKPOINT).to(device)
auto_generator = SamAutomaticMaskGenerator(
    sam, points_per_side=32, pred_iou_thresh=0.86,
    stability_score_thresh=0.92, min_mask_region_area=0,  # 0 bypasses OpenCV bridging bug
)

print("SAM loaded.")

In [ ]:
# Segmentation with caching
# Persistent mask label cache — avoids re-calling Claude on reruns
CACHE_PATH = f'{CHECKPOINTS_DIR}/mask_labels_cache.json'
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, 'r') as f:
        mask_cache = json.load(f)
    print(f"Loaded mask cache: {len(mask_cache)} images")
else:
    mask_cache = {}

def save_cache():
    with open(CACHE_PATH, 'w') as f:
        json.dump(mask_cache, f, indent=2)

# Test segmentation
raw_files = [r for r, e in pairs]
test_idx = 0
test_img = Image.open(os.path.join(RAW_DIR, raw_files[test_idx])).convert('RGB')
test_img_resized = resize_for_sam(test_img)

print(f"Testing on: {raw_files[test_idx]} ({test_img_resized.size})")
segments = segment_and_label(
    test_img_resized, auto_generator, client,
    cache=mask_cache, image_id=raw_files[test_idx]
)
save_cache()
visualize_segments(test_img_resized, segments)
print(f"Labels: {[s['label'] for s in segments]}")

## 3. Regional Editing: Text → Masked Parameter Application

Combine SAM masks + Claude prompt parsing + feathered blending.
User says "brighten the face and cool the sky" → parsed to per-region params → blended result.

In [ ]:
# Regional prompt parsing + end-to-end test
def parse_regional_prompt(prompt, available_labels, client):
    """Claude interprets text into [{region, params}] constrained to detected labels."""
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=800,
        system=f"""You parse photo editing instructions into regional edits.
Available regions: {json.dumps(available_labels)}

Output ONLY valid JSON: a list of edits. Each has:
- "region": one of the available labels, or "global"
- "params": object with Lightroom parameter names and numeric values

Ranges: exposure [-5,5], contrast/shadows/highlights [-100,100],
temperature [-100,100], clarity/vibrance/saturation [-100,100].
Also supports tone_curve (blacks/shadows/midtones/highlights/whites)
and color_mixer (reds/oranges/yellows/greens/aquas/blues/purples/magentas with h/s/l).

Be precise. "slightly brighten" → exposure +0.3, not +2.0.""",
        messages=[{"role": "user", "content": prompt}],
    )
    result = extract_json(response.content[0].text)
    if result and isinstance(result, list):
        return result
    return []


# End-to-end test
renderer = DeterministicRenderer()
available_labels = list(set(s['label'] for s in segments))

test_prompts = [
    "brighten the subject and add warmth to their skin",
    "cool the sky and make it more vivid",
    "lift the shadows in the background and reduce clarity",
]

for prompt in test_prompts:
    print(f'\n{"=" * 60}')
    print(f'Prompt: "{prompt}"')
    regional_edits = parse_regional_prompt(prompt, available_labels, client)
    print(f"Parsed: {json.dumps(regional_edits, indent=2)}")

    result = apply_regional_edits(test_img_resized, segments, regional_edits, renderer)
    show_before_after(test_img_resized, result, title=prompt[:50])

In [ ]:
# Mathematical style extraction across all pairs
from collections import defaultdict
from tqdm import tqdm

class RegionalStyleExtractor:
    """Extracts mathematical parameter shifts from raw/edited pairs per-region."""
    def __init__(self, min_pixels=1000):
        self.min_pixels = min_pixels

    def extract(self, raw_pixels, edt_pixels):
        """Derive shifts using medians (robust to segmentation edge noise).
        raw_pixels, edt_pixels: (N, 3) float32 [0, 1]."""
        if len(raw_pixels) < self.min_pixels:
            return {}

        raw_lum = 0.299*raw_pixels[:,0] + 0.587*raw_pixels[:,1] + 0.114*raw_pixels[:,2]
        edt_lum = 0.299*edt_pixels[:,0] + 0.587*edt_pixels[:,1] + 0.114*edt_pixels[:,2]

        raw_med = np.median(raw_lum) + 1e-5
        edt_med = np.median(edt_lum) + 1e-5

        exposure = float(np.log2(edt_med / raw_med))
        contrast = float((np.std(edt_lum) / (np.std(raw_lum) + 1e-5) - 1.0) * 100)

        raw_rb = (raw_pixels[:,0] + 1e-5) / (raw_pixels[:,2] + 1e-5)
        edt_rb = (edt_pixels[:,0] + 1e-5) / (edt_pixels[:,2] + 1e-5)
        temperature = float(np.median(edt_rb - raw_rb) * 100)

        shadow_mask = raw_lum < np.percentile(raw_lum, 25)
        shadows = float((np.median(edt_lum[shadow_mask]) - np.median(raw_lum[shadow_mask])) * 100) if shadow_mask.any() else 0.0

        hl_mask = raw_lum > np.percentile(raw_lum, 75)
        highlights = float((np.median(edt_lum[hl_mask]) - np.median(raw_lum[hl_mask])) * 100) if hl_mask.any() else 0.0

        raw_sat = np.max(raw_pixels, axis=1) - np.min(raw_pixels, axis=1)
        edt_sat = np.max(edt_pixels, axis=1) - np.min(edt_pixels, axis=1)
        saturation = float((np.median(edt_sat) - np.median(raw_sat)) * 100)

        return {
            "exposure": round(exposure, 3),
            "contrast": round(contrast, 2),
            "temperature": round(temperature, 2),
            "shadows": round(shadows, 2),
            "highlights": round(highlights, 2),
            "saturation": round(saturation, 2),
        }

    def process_pair(self, raw_img, edt_img, segments):
        """Extract per-region parameters for one image pair."""
        raw_np = np.array(raw_img).astype(np.float32) / 255.0
        edt_np = np.array(edt_img).astype(np.float32) / 255.0
        if raw_np.shape != edt_np.shape:
            edt_np = cv2.resize(edt_np, (raw_np.shape[1], raw_np.shape[0]))

        profiles = {}
        for seg in segments:
            mask = seg['mask']
            if mask.shape[:2] != raw_np.shape[:2]:
                mask = np.array(Image.fromarray(mask.astype(np.uint8)).resize(
                    (raw_np.shape[1], raw_np.shape[0]), Image.NEAREST
                )).astype(bool)
            else:
                mask = mask.astype(bool)

            params = self.extract(raw_np[mask], edt_np[mask])
            if params:
                profiles[seg['label']] = params
        return profiles


# Run extraction across all pairs
extractor = RegionalStyleExtractor(min_pixels=1000)
master_stats = defaultdict(list)
edited_files = [e for r, e in pairs]

print(f"Extracting mathematical style from {len(pairs)} pairs...")
for i in tqdm(range(len(pairs))):
    raw_f, edited_f = pairs[i]
    try:
        r_img = Image.open(os.path.join(RAW_DIR, raw_f)).convert("RGB")
        e_img = Image.open(os.path.join(EDITED_DIR, edited_f)).convert("RGB")
        r_img_eval = resize_for_sam(r_img)

        segs = segment_and_label(r_img_eval, auto_generator, client,
                                  cache=mask_cache, image_id=raw_f)
        pair_profiles = extractor.process_pair(r_img_eval, e_img, segs)

        for label, params in pair_profiles.items():
            master_stats[label].append(params)
    except Exception as e:
        print(f"\nSkipped {raw_f}: {e}")

save_cache()  # Persist any new LLM labels

# Aggregate via medians
math_profile = {}
for label, occurrences in master_stats.items():
    if len(occurrences) < len(pairs) * 0.05:  # min 5% representation
        continue
    params = {}
    for key in occurrences[0].keys():
        values = [o[key] for o in occurrences if key in o]
        params[key] = round(float(np.median(values)), 3)
    math_profile[label] = {"parameters": params, "sample_size": len(occurrences)}

path = save_style_profile(math_profile, 'simon_math_profile.json')
print(f"\nMath profile saved: {path}")
print(f"Regions profiled: {list(math_profile.keys())}")

In [ ]:
# Validation table: display the extracted profile as a clean table
import pandas as pd

data = []
for region, d in math_profile.items():
    row = {'Region': region, 'Samples': d['sample_size']}
    row.update(d['parameters'])
    data.append(row)

df = pd.DataFrame(data).set_index('Region').sort_values('Samples', ascending=False)
display(df.style.format(precision=2).background_gradient(cmap='RdYlGn', axis=None, subset=df.columns[1:]))

print(f"\nPipeline 02 complete.")
print("  → style_profiles_llm.json (LLM-estimated per-scene profiles)")
print("  → simon_math_profile.json (mathematical per-region profiles)")
print("  → mask_labels_cache.json (cached SAM + Claude labels)")
print("\nNext: Pipeline 03 (Advanced Color Science Extraction)")